# Black-Scholes Option Pricer — Methodology Notebook

**Author:** tashrifulkabir34-lang  
**Purpose:** Interactive walkthrough of pricing, Greeks, IV solving, strategies, and stress testing.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.black_scholes import BlackScholesModel, OptionParams
from src.greeks import GreeksCalculator
from src.implied_vol import ImpliedVolatilitySolver
from src.strategies import OptionStrategy
from src.scenarios import ScenarioAnalyzer, HISTORICAL_STRESS_SCENARIOS

plt.style.use('dark_background')
plt.rcParams['figure.dpi'] = 120
print('All modules loaded successfully.')

## 1. Black-Scholes Pricing

The BS formula for a European call with continuous dividend yield:

$$C = S e^{-qT} \Phi(d_1) - K e^{-rT} \Phi(d_2)$$

where $d_1 = \frac{\ln(S/K) + (r-q+\sigma^2/2)T}{\sigma\sqrt{T}}$, $d_2 = d_1 - \sigma\sqrt{T}$

In [ ]:
# Base parameters
S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20

call_params = OptionParams(S=S, K=K, T=T, r=r, sigma=sigma, option_type='call')
put_params  = OptionParams(S=S, K=K, T=T, r=r, sigma=sigma, option_type='put')

call_model = BlackScholesModel(call_params)
put_model  = BlackScholesModel(put_params)

print(f'{'='*45}')
print(f'  S={S} | K={K} | T={T}yr | r={r*100:.0f}% | σ={sigma*100:.0f}%')
print(f'{'='*45}')
print(f'  d1 = {call_model.d1:.6f}')
print(f'  d2 = {call_model.d2:.6f}')
print(f'  Call Price    = ${call_model.price():.4f}')
print(f'  Put  Price    = ${put_model.price():.4f}')
print(f'  PCP Residual  = {call_model.put_call_parity_check():.2e}')

## 2. Price Sensitivity to Spot and Volatility

In [ ]:
S_range   = np.linspace(60, 140, 200)
vol_range = [0.10, 0.20, 0.30, 0.40]
colors    = ['#64ffda', '#c3a6ff', '#f4c87e', '#ff6b6b']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for vol, col in zip(vol_range, colors):
    c_prices, p_prices = [], []
    for s in S_range:
        cp = OptionParams(S=s, K=K, T=T, r=r, sigma=vol, option_type='call')
        pp = OptionParams(S=s, K=K, T=T, r=r, sigma=vol, option_type='put')
        c_prices.append(BlackScholesModel(cp).price())
        p_prices.append(BlackScholesModel(pp).price())
    axes[0].plot(S_range, c_prices, color=col, label=f'σ={vol*100:.0f}%')
    axes[1].plot(S_range, p_prices, color=col, label=f'σ={vol*100:.0f}%')

for ax, title in zip(axes, ['Call Price vs Spot', 'Put Price vs Spot']):
    ax.axvline(K, color='white', linestyle='--', alpha=0.4, label=f'K={K}')
    ax.set_xlabel('Spot Price S')
    ax.set_ylabel('Option Price ($)')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 3. Greeks Profile

In [ ]:
sa = ScenarioAnalyzer(S=S, K=K, T=T, r=r, sigma=sigma)
sens = sa.spot_sensitivity(S_range)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
labels = ['price', 'delta', 'gamma', 'theta', 'vega', 'pnl']
titles = ['Option Price', 'Delta Δ', 'Gamma Γ', 'Theta Θ/day', 'Vega V/1%', 'P&L']
cols   = ['#64ffda','#7ec8e3','#f4c87e','#ff6b6b','#c3a6ff','#64ffda']

for ax, lbl, ttl, col in zip(axes.flatten(), labels, titles, cols):
    ax.plot(S_range, sens[lbl], color=col, linewidth=2)
    ax.axvline(S, color='white', linestyle='--', alpha=0.4)
    ax.axvline(K, color='#f4c87e', linestyle=':', alpha=0.4)
    ax.set_title(ttl)
    ax.set_xlabel('Spot Price')
    ax.grid(alpha=0.2)

plt.suptitle('Greeks Profile vs Spot Price', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Implied Volatility — Round-Trip Accuracy

In [ ]:
sigma_range_test = np.arange(0.05, 1.05, 0.05)
solver = ImpliedVolatilitySolver(S=S, K=K, T=T, r=r)
errors, iters = [], []

for sig_true in sigma_range_test:
    params = OptionParams(S=S, K=K, T=T, r=r, sigma=sig_true)
    mkt_price = BlackScholesModel(params).price()
    diag = solver.solve(mkt_price, return_diagnostics=True)
    errors.append(abs(diag['sigma'] - sig_true))
    iters.append(diag['iterations'] or 0)

df_iv = pd.DataFrame({
    'True σ (%)': sigma_range_test * 100,
    'Absolute Error': errors,
    'NR Iterations': iters
})
print('IV Round-Trip Accuracy:')
print(df_iv.to_string(index=False, float_format=lambda x: f'{x:.2e}' if x < 0.01 else f'{x:.2f}'))
print(f'\nMax error: {max(errors):.2e}  |  Mean iterations: {np.mean(iters):.1f}')

## 5. Strategy Payoff Diagrams

In [ ]:
S_plot = np.linspace(60, 140, 300)
strats = [
    ('Long Straddle',    OptionStrategy.long_straddle(S, K, T, r, sigma)),
    ('Bull Call Spread', OptionStrategy.bull_call_spread(S, K, K*1.10, T, r, sigma)),
    ('Iron Condor',      OptionStrategy.iron_condor(S, K*0.85, K*0.95, K*1.05, K*1.15, T, r, sigma)),
    ('Long Butterfly',   OptionStrategy.long_butterfly(S, K*0.90, K, K*1.10, T, r, sigma)),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (name, strat) in zip(axes.flatten(), strats):
    pnl = strat.expiry_pnl(S_plot)
    be  = strat.breakeven_points(S_plot)
    ax.plot(S_plot, pnl, '#64ffda', linewidth=2.5)
    ax.fill_between(S_plot, pnl, 0, where=(pnl>=0), alpha=0.12, color='#64ffda')
    ax.fill_between(S_plot, pnl, 0, where=(pnl<0),  alpha=0.12, color='#ff6b6b')
    ax.axhline(0, color='white', linewidth=0.8, alpha=0.5)
    ax.axvline(S, color='#f4c87e', linestyle='--', alpha=0.6)
    for b in be:
        ax.axvline(b, color='#ff6b6b', linestyle=':', alpha=0.5)
    be_str = ', '.join([f'${b:.1f}' for b in be]) or 'N/A'
    ax.set_title(f'{name}\nNet Premium: ${strat.net_premium:.2f} | Break-evens: {be_str}', fontsize=9)
    ax.set_xlabel('Spot at Expiry')
    ax.set_ylabel('P&L ($)')
    ax.grid(alpha=0.2)

plt.suptitle('Option Strategy Payoff Diagrams at Expiry', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Historical Stress Tests (Out-of-Sample)

In [ ]:
sa = ScenarioAnalyzer(S=S, K=K, T=T, r=r, sigma=sigma, option_type='call')
results = sa.stress_test()
df = pd.DataFrame(results)

print('ATM Call — Historical Stress Test Results:')
print(df[['Scenario','New Spot','New σ','New Price','P&L ($)','P&L (%)']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#64ffda' if x >= 0 else '#ff6b6b' for x in df['P&L ($)']]
ax.barh(df['Scenario'], df['P&L ($)'], color=colors, alpha=0.85)
ax.axvline(0, color='white', linewidth=1)
ax.set_xlabel('P&L ($)')
ax.set_title('Option P&L Under Historical Stress Scenarios')
ax.grid(alpha=0.2, axis='x')
plt.tight_layout()
plt.show()

## 7. P&L Heatmap: Spot × Volatility

In [ ]:
S_heat     = np.linspace(70, 130, 40)
sigma_heat = np.linspace(0.05, 0.50, 40)
heatmap    = sa.spot_vol_heatmap(S_heat, sigma_heat)

fig, ax = plt.subplots(figsize=(10, 7))
vmax = np.abs(heatmap).max()
im = ax.imshow(
    heatmap, aspect='auto', origin='lower',
    extent=[sigma_heat[0]*100, sigma_heat[-1]*100, S_heat[0], S_heat[-1]],
    cmap='RdYlGn', vmin=-vmax, vmax=vmax
)
ax.axhline(S, color='white', linestyle='--', linewidth=1, alpha=0.6, label=f'Current S={S}')
ax.axvline(sigma*100, color='white', linestyle=':', linewidth=1, alpha=0.6, label=f'Current σ={sigma*100:.0f}%')
plt.colorbar(im, ax=ax, label='P&L ($)')
ax.set_xlabel('Implied Volatility (%)')
ax.set_ylabel('Spot Price')
ax.set_title('Call Option P&L: Spot × Volatility Heatmap')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()